# convT-kernel-axis-swap — faded example 2: Complete the flipped, swapped kernel for the full-conv identity

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-kernel-axis-swap`. The last cell reports your progress on the `CNN: ConvT kernel axis swap` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT kernel axis swap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-kernel-axis-swap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-kernel-axis-swap"
DD_SUBTOPIC = "CNN: ConvT kernel axis swap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A stride-1 `ConvTranspose2d` equals a *full* `Conv2d` whose kernel has been channel-axis-swapped `(IC, OC, KH, KW) -> (OC, IC, KH, KW)` and then spatially flipped on both kernel axes. The swap fixes the layout difference; the flip turns correlation into the convolution the transpose performs.

## Faded exercise 2

### Faded — reconstruct ConvTranspose2d from F.conv2d

Implement `convT2d_via_full_conv(x, w_convT)`. The kernel `w_convT` is in transposed-conv layout `(IC, OC, KH, KW)`. Build the conv-layout kernel needed for the full-convolution identity, then call `F.conv2d` with `padding = K - 1`. Complete the one line that builds the flipped, axis-swapped conv kernel.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn.functional as F
def convT2d_via_full_conv(x: Tensor, w_convT: Tensor) -> Tensor:
    k = w_convT.shape[-1]
    w_conv = t.flip(rearrange(w_convT, 'i o kh kw -> o i kh kw'), dims=[2, 3])
    return F.conv2d(x, w_conv, padding=k - 1)


def _test():
    t.manual_seed(0)
    ic, oc, k = 3, 2, 3
    w = t.randn(ic, oc, k, k)      # (IC, OC, KH, KW)
    x = t.randn(1, ic, 6, 6)
    ct = t.nn.ConvTranspose2d(ic, oc, k, bias=False)
    ct.weight.data.copy_(w)
    ref = ct(x)
    got = convT2d_via_full_conv(x, w)
    assert tuple(got.shape) == tuple(ref.shape), (tuple(got.shape), tuple(ref.shape))
    assert t.allclose(got, ref, atol=1e-5), 'full-conv reconstruction does not match ConvTranspose2d'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F
def convT2d_via_full_conv(x: Tensor, w_convT: Tensor) -> Tensor:
    k = w_convT.shape[-1]
    w_conv = t.flip(rearrange(w_convT, 'i o kh kw -> o i kh kw'), dims=[2, 3])
    return F.conv2d(x, w_conv, padding=k - 1)
```
</details>